In [3]:
import os
from dotenv import load_dotenv
from snowflake.snowpark import Session
from snowflake.snowpark import functions as F
from snowflake.core import Root, CreateMode
from snowflake.core.schema import Schema
from snowflake.core.table import Table, TableColumn

_ = load_dotenv(override=True)

connection_parameters = {
    "account": os.getenv("SNOWFLAKE_ACCOUNT"),
    "user": os.getenv("SNOWFLAKE_USER"),
    "password": os.getenv("SNOWFLAKE_PASSWORD"),
    "authenticator": "username_password_mfa",
    "role": os.getenv("SNOWFLAKE_ROLE"),
    "warehouse": os.getenv("SNOWFLAKE_WAREHOUSE"),
    "database": os.getenv("SNOWFLAKE_DATABASE"),
    "schema": "PUBLIC"
}

session = Session.builder.configs(connection_parameters).create()
root = Root(session)

DATABASE = os.getenv("SNOWFLAKE_DATABASE")
WAREHOUSE = os.getenv("SNOWFLAKE_WAREHOUSE")
SHARED_SCHEMA = "shared"
INTERNAL_STAGE = f"{SHARED_SCHEMA}.int_multimodal_data_files"

current_user = session.get_current_user()
print(f"Connected as {current_user}")
print(f"Welcome to the course! Let's start building.")

Connected as "ARSHAD18"
Welcome to the course! Let's start building.


In [5]:
MY_SCHEMA = current_user.strip('"').lower()

schema = Schema(name=MY_SCHEMA)
root.databases[DATABASE].schemas.create(schema, mode=CreateMode.if_not_exists)
session.use_schema(MY_SCHEMA)

print(f"Using schema: {DATABASE}.{MY_SCHEMA}")

Using schema: MULTIMODAL_DB.arshad18


In [6]:
audio_transcripts_table = Table(
    name="audio_transcripts",
    columns=[
        TableColumn(name="meeting_id", datatype="STRING"),
        TableColumn(name="meeting_part", datatype="STRING"),
        TableColumn(name="audio_file_path", datatype="STRING"),
        TableColumn(name="transcript_text", datatype="STRING"),
        TableColumn(name="audio_duration", datatype="FLOAT"),
    ]
)

root.databases[DATABASE].schemas[MY_SCHEMA].tables.create(
    audio_transcripts_table,
    mode=CreateMode.or_replace
)

print("Created audio_transcripts table")

print("Loading Whisper model...")

model = WhisperModel("base")

print("Whisper model loaded!")

meeting_id = "ES2008"

meeting_parts = ["a", "b", "c", "d"]

Created audio_transcripts table
Loading Whisper model...
Whisper model loaded!


In [9]:
session.file.put(
    "C:/Users/arsha/Downloads/meeting1.wav",
    "@PUBLIC.MY_AUDIO_STAGE",
    auto_compress=False
)

[PutResult(source='meeting1.wav', target='meeting1.wav', source_size=5289194, target_size=5289200, source_compression='NONE', target_compression='NONE', status='UPLOADED', message='')]

In [10]:
for part in meeting_parts:

    print(f"\nProcessing part {part}...")

    # Snowflake stage audio path
    audio_path = "@PUBLIC.MY_AUDIO_STAGE/meeting1.wav"


    print("Downloading audio file from Snowflake stage...")

    # Download file locally
    session.file.get(audio_path, ".")

    # Local file path
    local_audio_path = "./meeting1.wav"

    print("Transcribing audio using Whisper...")

    # Transcribe audio
    segments, info = model.transcribe(local_audio_path)

    # Combine transcript
    transcript_text = ""

    for segment in segments:
        transcript_text += segment.text + " "

    print("Transcription completed!")

    # Escape quotes
    transcript_text = transcript_text.replace("'", "''")

    # Insert into Snowflake
    insert_query = f"""
    INSERT INTO audio_transcripts
    SELECT
        '{meeting_id}' AS meeting_id,
        '{part}' AS meeting_part,
        '{audio_path}' AS audio_file_path,
        '{transcript_text}' AS transcript_text,
        {info.duration} AS audio_duration
    """

    session.sql(insert_query).collect()

    print(f"Inserted transcript for meeting {meeting_id}{part}")


Processing part a...
Transcribing audio using Whisper...
Transcription completed!
Inserted transcript for meeting ES2008a

Processing part b...
Transcribing audio using Whisper...
Transcription completed!
Inserted transcript for meeting ES2008b

Processing part c...
Transcribing audio using Whisper...
Transcription completed!
Inserted transcript for meeting ES2008c

Processing part d...
Transcribing audio using Whisper...
Transcription completed!
Inserted transcript for meeting ES2008d


In [12]:
from faster_whisper import WhisperModel

model = WhisperModel("base")

segments, info = model.transcribe("meeting1.wav")

for segment in segments:
    print(segment.text)